# Document Loader Base Interfaces Reference

Developer-facing statements defined in `langchain_core.document_loaders.base`.

# `BaseLoader: ABC`

Interface base class for document loaders.

Implementations should prefer generator-based lazy loading to avoid loading every document into memory at once. In this pinned version, the class has no abstract methods, but concrete loaders should normally override `lazy_load`.

## Methods

### `load`

Eagerly loads all documents by consuming `lazy_load`. Subclasses should not override this method.

```python
load(
    self,
) -> list[Document] # Loaded documents
```

### `aload`

Asynchronously loads all documents by consuming `alazy_load`.

```python
async aload(
    self,
) -> list[Document] # Loaded documents
```

### `load_and_split`

Loads documents and splits them into chunks.

This method should not be overridden and should be considered deprecated.

```python
load_and_split(
    self,
    text_splitter: TextSplitter | None = None, # Splitter to use; defaults to RecursiveCharacterTextSplitter
) -> list[Document] # Split document chunks
```

Raises `ImportError` when no splitter is supplied and `langchain-text-splitters` is unavailable.

### `lazy_load`

Returns an iterator that lazily produces documents.

```python
lazy_load(
    self,
) -> Iterator[Document] # Lazily loaded documents
```

Concrete loaders should override this method. For backward compatibility, if a subclass overrides `load` instead, this method returns an iterator over that result. Otherwise, it raises `NotImplementedError`.

### `alazy_load`

Asynchronously yields documents from `lazy_load`.

```python
async alazy_load(
    self,
) -> AsyncIterator[Document] # Lazily loaded documents
```

The synchronous iterator and each `next` call are executed through `run_in_executor`.

---

# `BaseBlobParser: ABC`

Abstract interface for parsers that convert a `Blob` into one or more `Document` objects.

A concrete subclass must implement `lazy_parse`.

## Required subclass hooks

### `lazy_parse`

Lazily parses a blob into documents.

```python
lazy_parse(
    self,
    blob: Blob, # Blob to parse
) -> Iterator[Document] # Parsed documents
```

## Methods

### `parse`

Eagerly parses a blob by consuming `lazy_parse`.

```python
parse(
    self,
    blob: Blob, # Blob to parse
) -> list[Document] # Parsed documents
```

This convenience method is intended mainly for interactive development. Production applications should prefer `lazy_parse`, and subclasses should generally not override `parse`.

In [ ]:
# !pip install langchain-core#Install LangChain Core if it is not already installed

from collections.abc import Iterator#Import the Iterator type for lazy document generation
from langchain_core.document_loaders.base import BaseBlobParser, BaseLoader#Import the base loader and blob parser classes
from langchain_core.documents import Blob, Document#Import Blob for raw data and Document for parsed content


class TextLineParser(BaseBlobParser):#Create a parser that converts each text line into a Document

    def lazy_parse(self, blob: Blob) -> Iterator[Document]:#Implement the required lazy_parse method
        text = blob.as_string()#Convert the Blob data into a string
        source = blob.source or "unknown"#Get the source name or use a default value

        for line_number, line in enumerate(text.splitlines(), start=1):#Process every line with its line number
            cleaned_line = line.strip()#Remove spaces from the beginning and end of the line

            if cleaned_line:#Ignore empty lines
                metadata = {"source": source, "line_number": line_number}#Create metadata for the current line
                document = Document(page_content=cleaned_line, metadata=metadata)#Create a Document from the line
                yield document#Return the Document lazily


class InMemoryTextLoader(BaseLoader):#Create a loader for text stored directly in memory

    def __init__(self, text: str, source: str = "memory.txt") -> None:#Initialize the loader
        self.text = text#Store the supplied text
        self.source = source#Store the source name
        self.parser = TextLineParser()#Create the custom blob parser

    def lazy_load(self) -> Iterator[Document]:#Implement lazy document loading
        blob = Blob.from_data(#Create a Blob from the supplied text
            data=self.text,#Provide the raw text data
            mime_type="text/plain",#Specify that the Blob contains plain text
            metadata={"source": self.source},#Attach the source name as metadata
        )

        yield from self.parser.lazy_parse(blob)#Parse and lazily return the Documents


sample_text = "Python is easy to learn.\\n\\nLangChain helps build LLM applications.\\nCallbacks monitor model execution."#Create sample multiline text

loader = InMemoryTextLoader(#Create the custom document loader
    text=sample_text,#Pass the sample text
    source="langchain_notes.txt",#Provide a source filename
)

documents = loader.load()#Load all Documents by consuming lazy_load

print(f"Total documents: {len(documents)}")#Display the number of loaded Documents
print("-" * 50)#Display a separator line

for document in documents:#Iterate through every loaded Document
    print(f"Content: {document.page_content}")#Display the Document content
    print(f"Metadata: {document.metadata}")#Display the Document metadata
    print("-" * 50)#Display a separator after each Document